# PersonaPlex — RunPod RTX 5090 Deployment Notebook

This notebook deploys **[PersonaPlex](https://github.com/NVIDIA/personaplex)** — NVIDIA's real-time, full-duplex
speech-to-speech model with persona/voice control (built on [Moshi](https://arxiv.org/abs/2410.00037)) — on a
**fresh RunPod pod with an RTX 5090 GPU**.

It is derived directly from the repository's own `README.md`, `client/README.md`, `moshi/pyproject.toml`,
`moshi/requirements.txt`, and the server/offline entrypoints (`moshi/moshi/server.py`, `moshi/moshi/offline.py`,
`moshi/moshi/utils/connection.py`, `moshi/moshi/models/loaders.py`). No steps are invented — every command below
maps to something the repo actually does or documents.

## What this notebook does

1. Verifies the RunPod environment, GPU and CUDA.
2. Sets up persistent storage on the RunPod volume.
3. Installs system + Python dependencies (including the **Blackwell/RTX 5090‑specific PyTorch build**).
4. Clones the repository (skipped automatically if you already uploaded it).
5. Configures Hugging Face authentication and downloads the model weights, tokenizer, voices and web UI assets.
6. Starts the PersonaPlex backend server (which also serves the prebuilt web UI — no separate frontend build is
   required for normal use).
7. Verifies the deployment with an automated offline inference smoke test and an HTTP check.
8. Documents GPU optimization notes and troubleshooting steps.

## What PersonaPlex does *not* have (so these checklist items are intentionally empty)

- **No database** of any kind — there is nothing to initialize.
- **No separate "API server"** — the single aiohttp server in `moshi/moshi/server.py` exposes both the WebSocket
  endpoint (`/api/chat`) and the static web UI on one port.
- **No Gradio/Streamlit app** — the only Gradio dependency is `gradio.networking.setup_tunnel`, an *optional*
  public-URL tunnel (`--gradio-tunnel`), not a Gradio UI. RunPod's own HTTP port proxy makes this unnecessary here.
- **No required "configuration file" to generate** — runtime behavior is controlled entirely by CLI flags and the
  HF-hosted `config.json` that ships with the model weights.

## Things this notebook *cannot* automate (you must do these yourself)

1. **Accept the NVIDIA Open Model License** for the gated model repo
   [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1) — log into Hugging Face in a
   browser and click "Agree and access repository". No script can click this for you.
2. **Create a Hugging Face access token** for that same account and have it ready to paste into the auth cell
   below.
3. **Expose the server port (default `8998`) as an HTTP Service** from the RunPod pod's *Connect* page (RunPod
   console action) so the proxy URL works from outside the pod.
4. **Grant microphone permission** in your browser when you open the live web UI — this is a per-user browser
   security prompt.

Run the cells **top to bottom**. The only cell you must intentionally run out of the normal flow is the final
**"Stop the server"** cleanup cell — leave it for whenever you actually want to shut the server down.

## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).

In [1]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")

Platform: Linux-6.8.0-60-generic-x86_64-with-glibc2.39
Python: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]
Python version OK.


In [2]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi

Thu Jul  2 13:50:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.133.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:83:00.0 Off |                  N/A |
| 37%   37C    P8             25W /  600W |       2MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Persistent storage setup (RunPod volume)

RunPod mounts your persistent Network Volume at **`/workspace`**. Anything written there survives pod
stop/start (model weights are multiple GB, so re-downloading them on every restart would be wasteful).
If `/workspace` isn't present (e.g. you're running this notebook somewhere else), we fall back to the home
directory so the notebook still works end-to-end.

We also point Hugging Face's cache (`HF_HOME`) at the persistent volume so `huggingface_hub` downloads
(triggered both by this notebook and internally by `moshi/moshi/server.py` / `moshi/moshi/offline.py`) are
cached once and reused across restarts.

In [3]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-original-code-streaming-prompt-with-tools-calling-v2.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.server

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM, comfortably enough for this 7B model in bf16 -- CPU offload should not be
# needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package, installed below anyway).
USE_CPU_OFFLOAD = False

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE   :", WORKSPACE)
print("REPO_DIR    :", REPO_DIR)
print("HF_HOME     :", os.environ["HF_HOME"])
print("USE_APP_TLS :", USE_APP_TLS)

WORKSPACE   : /workspace
REPO_DIR    : /workspace/personaplex
HF_HOME     : /workspace/.cache/huggingface
USE_APP_TLS : False


## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.

In [4]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
debconf: delaying package configuration, since apt-utils is not installed
(Reading database ... 49062 files and directories currently installed.)
Preparing to unpack .../ca-certificates_20260601~24.04.1_all.deb ...
Unpacking ca-certificates (20260601~24.04.1) over (20240203) ...
Selecting previously unselected package libopus-dev:amd64.
Preparing to unpack .../libopus-dev_1.4-1build1_amd64.deb ...
Unpacking libopus-dev:amd64 (1.4-1build1) ...
Setting up libopus-dev:amd64 (1.4-1build1) ...
Setting up ca-certificates (20260601~24.04.1) ...
Updating certificates in /etc/ssl/certs...
rehash: warning: skipping ca-certificates.crt,it does not contain exactly one certificate or CRL
14 added, 39 removed; done.
Processing triggers for ca-certificates (20260601~24.04.1) ...
Updating cert

## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project (e.g. you uploaded your own copy of this repo to the volume
beforehand), it is cloned fresh from GitHub. If it's already there, cloning is skipped automatically — this
cell is safe to re-run on every pod restart.

In [5]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


Cloning into '/workspace/personaplex'...
Updating files:  87% (90/103)

Cloned into /workspace/personaplex.


Updating files: 100% (103/103), done.


## 4b. Live web-search tool-calling for Helium

Adds the ability for the live voice assistant to answer time-sensitive questions (prices, weather,
news, sports scores, flight status, etc.) by searching the web -- **PersonaPlex's own Helium model
remains the only model in the pipeline**; no second LLM is ever called. Helium decides on its own,
via an appended system-prompt instruction, whether a question needs live information; if so it
replies with exactly `SEARCH: <query>`, and the code below intercepts that before its audio reaches
you, performs the search, and feeds the results back into the *same* Helium session (conversation
memory preserved) so it generates the real, freshly-spoken answer.

**Why this needs to patch `server.py` and `lm.py`, not just this notebook:** PersonaPlex is a
full-duplex, real-time speech model (Sections 10-14 launch `python -m moshi.server`, a live
WebSocket that generates audio and text jointly, frame by frame). There is no "send a prompt, get a
response" boundary to hook into from outside, and the client currently only sends *audio* to the
server (no text-input channel). The cells below patch the **freshly cloned repo** at `REPO_DIR`
(from Section 4) *before* it gets `pip install`-ed in Section 5, so the patched code is what actually
runs. Nothing here modifies the live web UI (`client/`) -- only how the already-running server
handles its own generated output.

**What to expect while it's running:**
- Ordinary questions get no perceptible extra delay and never trigger a search (check the server log
  to confirm).
- A live-info question causes a short **silent pause** (no filler audio) while the search runs and
  results are fed back to Helium -- typically a few seconds: the network fetch plus a bounded amount
  of time spent force-feeding the grounding text into Helium's context (capped by
  `PERSONAPLEX_SEARCH_PROMPT_MAX_CHARS`, since every character costs real GPU time to inject).
- Each search round also spends a bit of Helium's ~4-minute attention-window budget -- the same
  caveat Section 13b below already describes for the static "grounded responses" text prompt.
- Deciding when the model has *started a new utterance* (as opposed to a brief pause between words
  of the same utterance) is a heuristic, not a hard signal from the protocol -- see the
  `PERSONAPLEX_UTTERANCE_BOUNDARY_SILENCE_FRAMES` comment inside the `server.py` patch below if you
  ever see it mistime under real use. A mistimed *false arm* only costs a little extra buffering
  latency (harmless -- the buffered audio is still delivered intact); it cannot drop real speech.

Toggle the whole feature off with `ENABLE_WEB_SEARCH = True` below -- the patched server then takes
an early-exit path back to byte-for-byte original behavior.

---

### Prompt tuning (v2) -- getting Helium to actually trigger a search

PersonaPlex/Helium is a 7B real-time *speech* model, not a tool-calling-tuned chat LLM, and the
`search <query>` output format is not something it saw in training -- so whether it emits the trigger
at all is a prompt-engineering question, not a guarantee. Two changes make it as likely as possible:

1. **Instruction phrasing matches training style.** `TOOL_INSTRUCTIONS` in the search_tools.py cell is
   now written as a natural, second-person, spoken instruction (like the README "Prompting Guide"
   examples), not a machine API spec, and is **prepended before** your role prompt.
2. **Colon-free, word-boundary trigger detection.** In speech the model says "search gold price", never
   a written "SEARCH:". Detection (`match_search_prefix` / `needs_search_response`) is now
   case-insensitive, ignores punctuation, and is word-boundary aware ("searching" is not the trigger).

**All tuning lives in one cell.** To A/B test wording, edit `TOOL_INSTRUCTIONS` (or `SEARCH_TRIGGER`)
inside the search_tools.py cell below, then: re-run that cell -> run the server-upgrade cell -> **stop
the server (Section 17) and start it again (Section 10)** -> reconnect the browser. No other cell needs
re-running for prompt changes. (The running server reads the files under `REPO_DIR`, so a restart is
what picks up your edit.)

**Alternate wordings to try** (paste over `TOOL_INSTRUCTIONS`, keep it short):

- *More casual (pairs well with a "You enjoy having a good conversation." persona):*
  `"You can look things up when you need to. If the user asks about something current -- prices, weather, news, scores -- just say the word search and then what to look for, nothing else, and you'll get the answer to tell them. Otherwise, just talk normally."`
- *Very terse / imperative:*
  `"When the user asks about current facts like prices, weather, news, or scores, do not guess -- say only: search, then what to look up. Otherwise answer normally."`

**If it still refuses instead of searching:** that is the model declining the out-of-distribution
format, not a code bug (confirm in the server log -- a real trigger prints `tool call detected:
search ...`). Try a shorter persona, a shorter instruction, or an empty Text Prompt box to isolate
whether your role prompt is competing with the instruction. If no wording works reliably, the honest
conclusion is that this model won't do prompt-only tool-calling dependably, and a different mechanism
would be needed.

---

### Diagnosing why it refuses instead of searching (read the server log)

With `SEARCH_DEBUG` on (default), every browser connection now prints the *exact combined system
prompt the model received* to the server log. Run `print(tail(LOG_PATH, 200))` after a turn and look:

1. **Do you see a line `[search_tools] web-search ACTIVE. Combined system prompt = '...'`?**
   - **No** -> the running server is NOT executing this v2 code. You almost certainly did not restart
     the server after editing. Fix: re-run the search_tools.py cell + the server-upgrade cell, then
     **stop the server (Section 17) and start it again (Section 10)**. The `text prompt: ...` line by
     itself is not proof -- it only echoes what you typed, not the combined prompt.
   - **Yes, and it contains the tool instructions** -> the mechanism is active and correct. The model
     is simply choosing to refuse rather than emit the trigger. Continue to (2).
2. **Do you see `tool call detected: search ...`?**
   - **No** (only a spoken refusal) -> the model declined the out-of-distribution format for this
     wording. Try: one of the alternate `TOOL_INSTRUCTIONS` wordings above; an **empty Text Prompt
     box** (so the persona's "answer questions" instinct isn't competing); or `SEARCH_TRIGGER = "lookup"`.
   - **Yes** -> it worked; you should have heard a pause then a real answer.

**If, after trying the alternates and an empty persona, it still refuses:** that is the honest ceiling
of prompt-only tool-calling for this 7B speech model -- it was never trained to emit machine triggers,
and its refusal prior for live-data questions is strong. At that point the realistic options are:
(a) accept occasional/unreliable triggering, or (b) a different, non-prompt mechanism (e.g. detecting
the model's *refusal* itself and deriving the query from the topic it names) -- which is less clean and
less reliable at producing a good query, but does not depend on the model adopting a new output format.
Tell me which way you want to go if we reach this point.

---

### v3: handling a conversational filler before the trigger

In live testing the model complied but said **"One moment, search today gold price."** -- the correct
trigger and query, but with a short filler (*"One moment,"*) in front. Detection now accepts the trigger
anywhere within the first `SEARCH_PREFIX_MAX_WORDS` spoken words (default 6) of an utterance, and the
query is taken as whatever follows the trigger. Trade-off: the server holds back the first few words of
**every** normal answer until it can rule out a search command, so normal replies start with a small
delay (raise/lower `PERSONAPLEX_SEARCH_PREFIX_MAX_WORDS` to trade filler-tolerance against that delay).
`SEARCH_DECISION_MAX_FRAMES` was also raised (env `PERSONAPLEX_SEARCH_DECISION_MAX_FRAMES`, default 50)
so the buffering window is long enough to contain a filler plus the trigger.

With `SEARCH_DEBUG` on, the log now also prints a per-utterance verdict line, e.g.
`[search_tools] utterance verdict=match: '1 moment, search today gold price'` or `verdict=no: 'The gold
price ...'`, so you can see exactly how each utterance was classified.

---

### v4: fix the crash after the search runs

The first successful live search exposed a bug: feeding the results back into Helium reused the
pre-handshake `is_alive()` liveness check, which calls `ws.receive()` -- but the receive loop is
already awaiting `ws.receive()` during a live call, and aiohttp forbids two concurrent `receive()`
calls (`RuntimeError: Concurrent call to receive() is not allowed`), which killed the session right
after `injecting search-grounded prompt`. The v4 cell swaps in a `receive()`-free liveness check for
the injection path only. After this, the full loop -- detect -> search -> inject -> **speak the
answer** -- can complete in one session.

---

### v5: the reliability lever -- text sampling temperature

Once the crash was fixed, the remaining problem was pure **non-determinism**: for the exact same empty
prompt and question, the model said `search gold price` on one connection and refused ("I can't check
that, look it up on Google") on the next. The model samples its words at a temperature, so at the
default (0.7) the trigger is a coin-flip.

The config cell now exposes `TEXT_TEMPERATURE` (env `PERSONAPLEX_TEXT_TEMPERATURE`, default lowered to
**0.4**). Lower = the model follows the strong "reply only: search ..." instruction far more
consistently. Trade-off: normal conversation gets less varied/more repetitive. **Sweep it** -- try 0.4,
then 0.3, then 0.2 -- and pick the lowest value that still sounds natural. The log prints the active
value per connection: `text_temp=0.4 audio_temp=0.8`.

Note the honest caveat: lowering temperature amplifies whatever the model's *most likely* response is.
If, with the instruction in context, its top choice is the trigger, low temperature locks that in
(great). If its top choice is still the refusal, low temperature will lock in the *refusal* instead --
so if 0.4 makes it refuse every time, the answer is not "lower further", it's that this 7B speech model
resists this out-of-distribution behavior, and reliable tool-calling would need a different mechanism
(e.g. an ASR-based path). Temperature is the last lever inside the prompt-only approach.

---

### v6: refusal-detection (the chosen approach)

Prompt-only triggering was proven unreliable (the model refuses live-data questions at any temperature).
v6 stops fighting that and instead **harvests the refusal**: the system prompt now asks the model, when it
can't answer, to name the exact thing to look up as "search for &lt;topic&gt;" -- which is in-distribution
(the model already says "you could search for today's gold price online" on its own). The backend plays
the refusal normally, then `detect_search_query()` extracts the named topic, runs the search, and injects
the results so the model speaks the real answer.

What this means in practice:
- **UX:** you will hear the short refusal ("I don't have that, you could search for today's gold price"),
  then a brief pause, then the real answer. That's expected for this approach.
- **It's fuzzy by nature.** When the refusal names the topic ("search for X"), it works well. When the
  refusal names no topic ("look it up on Google") or a wrong one, no/again-wrong search happens -- the
  detection patterns in `search_tools.py` are deliberately conservative (better to miss than to search
  garbage) and are yours to tune.
- The log shows what was extracted: `[search_tools] extracted search query 'today's gold price' from
  utterance '...'`, then `tool call detected` / `injecting search-grounded prompt`.
- Normal (non-search) answers now have **no added latency** -- audio always plays live; detection happens
  only after an utterance finishes.

---

### v7: answer quality -- clean the results, and consider Tavily

The full loop now runs end-to-end (refusal -> detect -> search -> inject -> the model speaks). The last
issue is the *quality* of that spoken answer. In testing the model read out fragments of raw DuckDuckGo
snippets ("...reflect the current market conditions. You can check it out here...") instead of stating the
price. Two reasons and two fixes:

1. **The injected text was noisy.** v7's `build_search_prompt` now strips URLs and website call-to-action
   phrases, dedupes snippets, and tells the model explicitly to *state the actual value in one short
   sentence and not read out a website or a link*. (All tunable in the search_tools.py cell.)

2. **DuckDuckGo's free snippets are often just website descriptions and may not even contain the number.**
   This is the bigger limiter. For real answer quality, switch the provider to **Tavily**, which returns
   clean, answer-oriented content built for exactly this use:
   - Get a free API key at https://tavily.com (free tier is generous).
   - In the config cell (Section 4b) set `SEARCH_PROVIDER = "tavily"`, re-run it (it will prompt for the
     key), then restart the server.
   The rest of the pipeline is unchanged -- `search_web()` is provider-agnostic, so only that one setting
   moves.

**Honest expectation for this final step:** injecting results mid-conversation and having a 7B *speech*
model synthesize a clean spoken answer from them is the hardest part and inherently imperfect -- expect
occasional echoing or an odd phrasing even after these fixes. Better results (Tavily) + the cleaned prompt
give it the best shot; if a particular answer is poor, the `build_search_prompt` wording and
`PERSONAPLEX_SEARCH_PROMPT_MAX_CHARS` are the knobs to tune.

---

### v8: fix the "greeting instead of an answer" bug (injection format)

Symptom: the search ran and injected fine, but the model answered with a sign-off ("Hey, let me know if
you have any questions") instead of the price. Cause (good catch): the results were injected wrapped in
`<system> ... <system>` tags. In PersonaPlex's training a `<system>` block appears **only at the very
start of a conversation**, so injecting one mid-call reads to the model as *"a new conversation is
starting"* -- hence the greeting/sign-off.

v8 changes the injection to a natural **continuation of the model's own speech** (no `<system>` tags):
a short self-correction, the cleaned facts as silent context, then a lead-in the model continues out loud
as the answer. All three parts are tunable at the top of the search_tools.py cell:
- `INJECT_PREFIX` (default `"Oh wait, I actually just found that."`)
- `INJECT_LEADIN` (default `"So, to answer your question:"`) -- the model speaks its continuation of this.
- `INJECT_WITH_SYSTEM_TAGS=1` (env) to A/B test the old wrapped behavior.

If the model still doesn't cleanly answer, these are the knobs to try (e.g. a more directive lead-in like
`"The answer is"`, or a shorter `PERSONAPLEX_SEARCH_PROMPT_MAX_CHARS`). This mid-conversation
answer-synthesis is the hardest part for a 7B speech model and may remain imperfect, but injecting as
continuation rather than a system block is the structurally correct fix.

In [ ]:
# --- Section 4b config: web-search tool-calling ----------------------------------------
# ENABLE_WEB_SEARCH=False fully reverts the server to its original behavior (the patched
# handle_agent_frame() below takes an early-exit path when this is off).
ENABLE_WEB_SEARCH = True

# "duckduckgo" is free and needs no signup. "tavily" gives higher-quality, LLM-oriented
# snippets but requires a free API key from https://tavily.com.
SEARCH_PROVIDER = "tavily"

# Text-sampling temperature. The model samples its words, so at the default (0.7) it emits
# the "search ..." trigger only sometimes and refuses other times for the SAME question.
# Lowering this makes it follow the strong "reply only: search ..." instruction much more
# deterministically -- the single biggest lever on trigger reliability -- at the cost of
# less variety in normal conversation. Leave "" to keep the model default (0.7). If
# triggering is unreliable, sweep 0.5 -> 0.3 -> 0.1 and find the lowest value that still
# sounds natural for normal chat. The server reads this from its OWN environment (fixed
# when it was launched), so to change it: edit this value, re-run this cell, then restart
# the server (Section 17 to stop, Section 10 to start) so the new value reaches it.
TEXT_TEMPERATURE = "0.4"   # "" = model default (0.7)

os.environ["PERSONAPLEX_ENABLE_WEB_SEARCH"] = "1" if ENABLE_WEB_SEARCH else "0"
os.environ["PERSONAPLEX_SEARCH_PROVIDER"] = SEARCH_PROVIDER
os.environ["PERSONAPLEX_TEXT_TEMPERATURE"] = TEXT_TEMPERATURE

if SEARCH_PROVIDER == "tavily" and not os.environ.get("TAVILY_API_KEY"):
    from getpass import getpass
    os.environ["TAVILY_API_KEY"] = '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'

print(f"ENABLE_WEB_SEARCH = {ENABLE_WEB_SEARCH}")
print(f"SEARCH_PROVIDER   = {SEARCH_PROVIDER}")
print(f"TEXT_TEMPERATURE  = {TEXT_TEMPERATURE or '(model default 0.7)'}")


In [ ]:
import pathlib

# search_tools.py (v8) is the provider-agnostic search adapter AND the single place all
# prompt/detection/injection tuning lives. v8 fixes how search results are fed back to the
# model: NOT as a <system> block (which reads as "a new conversation is starting" mid-call
# and made the model reply with a greeting/sign-off), but as a natural CONTINUATION of the
# model's own speech (INJECT_PREFIX / facts / INJECT_LEADIN, all tunable at the top of the
# file). Edit anything below, re-run this cell, then restart the server.
SEARCH_TOOLS_SOURCE = '# SPDX-License-Identifier: MIT\n"""\nProvider-agnostic web search adapter for PersonaPlex\'s live tool-calling feature.\n\nThe rest of the pipeline only ever calls `search_web(query)`. To add a new provider,\nwrite a `_search_<name>(query, max_results)` adapter that returns the same shape\n(list of {"title", "snippet", "url"} dicts) and register it in `_PROVIDERS` -- no\nother code changes. Switch providers by setting PERSONAPLEX_SEARCH_PROVIDER (or\npassing `provider=` explicitly).\n\nHelium (the PersonaPlex LM) remains the only model that reasons about the user\'s\nquestion or writes any part of the final answer -- this module only fetches raw\nsearch snippets, it never calls another LLM.\n\n--- Prompt / trigger tuning lives ENTIRELY in this file (v2) ---\nBoth the wording Helium sees (TOOL_INSTRUCTIONS, build_tool_system_prompt) and how the\nserver recognizes Helium\'s spoken trigger (match_search_prefix, needs_search_response)\nare here, so you can iterate on prompt engineering by editing only this cell + restarting\nthe server -- no server.py re-patch needed.\n"""\n\nimport os\nimport re\nfrom typing import Dict, List, Optional\n\nSEARCH_PROVIDER = os.environ.get("PERSONAPLEX_SEARCH_PROVIDER", "duckduckgo")\nSEARCH_MAX_RESULTS = int(os.environ.get("PERSONAPLEX_SEARCH_MAX_RESULTS", "4"))\n# Keeps each mid-conversation injection bounded: forcing text into Helium\'s own text\n# channel costs ~1/12.5s of real GPU time per character-token, so a long prompt here\n# directly extends the silent pause the user hears during a search.\nSEARCH_PROMPT_MAX_CHARS = int(os.environ.get("PERSONAPLEX_SEARCH_PROMPT_MAX_CHARS", "500"))\n\n# --- How the search results are injected back into the live conversation (v8) ---------\n# KEY FIX: we do NOT wrap the injected facts in <system> ... <system> tags. In training a\n# <system> block appears only at the START of a conversation, so injecting one mid-call\n# reads as "a new conversation is starting" and the model replies with a greeting/sign-off\n# ("Hey, let me know if you have any questions") instead of answering. Instead we inject the\n# facts as a natural CONTINUATION of the model\'s own speech: a short self-correction, the\n# facts, then a lead-in that primes the model to *speak the answer* (its turn had just ended\n# with the refusal, so it needs a nudge to re-engage). All three pieces are tunable here.\n#   - INJECT_WITH_SYSTEM_TAGS=1 to A/B test the old <system>-wrapped behavior.\n#   - INJECT_PREFIX  : reframes the just-spoken refusal as "actually I found it".\n#   - INJECT_LEADIN  : an incomplete lead-in the model continues out loud as the answer.\nINJECT_WITH_SYSTEM_TAGS = os.environ.get("PERSONAPLEX_INJECT_WITH_SYSTEM_TAGS", "0") == "1"\nINJECT_PREFIX = os.environ.get("PERSONAPLEX_INJECT_PREFIX", "Oh wait, I actually just found that.")\nINJECT_LEADIN = os.environ.get("PERSONAPLEX_INJECT_LEADIN", "So, to answer your question:")\n\n# The single word Helium is asked to *speak* at the start of a turn when it needs live\n# info. Kept as a natural, easily-spoken word (not a machine token) to maximize the odds\n# a speech model actually says it. Detection below is case-insensitive and ignores any\n# punctuation, since in speech the model emits e.g. "search gold price" with no colon.\nSEARCH_TRIGGER = os.environ.get("PERSONAPLEX_SEARCH_TRIGGER", "search").strip().lower()\n\n# The model often speaks a short natural filler before the trigger, e.g.\n# "One moment, search today gold price." We therefore accept the trigger anywhere within\n# the first N spoken words of an utterance (not just as word #1). Larger N catches longer\n# fillers ("just give me a second, let me search ...") but adds latency to EVERY normal\n# answer, because the server holds back the first N words of every utterance until it can\n# rule out a search command. Tune to taste.\nSEARCH_PREFIX_MAX_WORDS = int(os.environ.get("PERSONAPLEX_SEARCH_PREFIX_MAX_WORDS", "6"))\n\n# When on, prints diagnostics (the exact combined system prompt, and trigger-match\n# decisions) into the server log so you can see EXACTLY what the model received and said.\n# On by default while tuning; set PERSONAPLEX_SEARCH_DEBUG=0 to quiet it.\nSEARCH_DEBUG = os.environ.get("PERSONAPLEX_SEARCH_DEBUG", "1") == "1"\n\n# --- Prompt Helium sees (v6: refusal-shaping) ----------------------------------------\n# We do NOT try to force the model to emit a machine trigger (that proved unreliable -- it\n# refuses at any temperature). Instead we lean INTO the model\'s natural, in-distribution\n# behavior: when it can\'t answer a live-data question it already tends to say things like\n# "you could search for today\'s gold price online". This prompt just shapes that refusal so\n# it *reliably names the exact topic* in a "search for <topic>" form, which the backend then\n# detects and extracts (see detect_search_query / _extract_refusal_topic below). Natural\n# refusal that names the topic is in-distribution, so this is a much better bet than the\n# trigger approach. Prepended before the user\'s role prompt.\nTOOL_INSTRUCTIONS = (\n    "You are a helpful voice assistant. When a question needs current, up-to-date "\n    "information you do not have -- like prices, weather, news, sports scores, or exchange "\n    "rates -- do not make something up. Briefly say you do not have it, and then clearly "\n    "tell the user the exact thing to look up, phrased as \\"search for <the specific topic>\\". "\n    "For example: \\"I do not have that, you could search for today\'s gold price.\\" For "\n    "everything else, answer normally."\n)\n\n\ndef build_tool_system_prompt(role_prompt: str) -> str:\n    """Prepend the tool-use instructions BEFORE whatever role prompt the user configured."""\n    role_prompt = (role_prompt or "").strip()\n    combined = f"{TOOL_INSTRUCTIONS} {role_prompt}" if role_prompt else TOOL_INSTRUCTIONS\n    if SEARCH_DEBUG:\n        # This one line per connection is the definitive proof the tool prompt is active:\n        # if you DON\'T see it in the server log, the server is not running this v2 code\n        # (re-run the search_tools + upgrade cells and restart the server).\n        print(f"[search_tools] web-search ACTIVE. Combined system prompt = {combined!r}", flush=True)\n    return combined\n\n\ndef _normalize(text: str) -> str:\n    """Lowercase and strip punctuation so spoken \'Search: gold\' == \'search gold\'."""\n    return re.sub(r"[^a-z0-9 ]", " ", (text or "").strip().lower())\n\n\ndef match_search_prefix(text: str) -> str:\n    """Classify accumulated agent text against the trigger word appearing within the first\n    SEARCH_PREFIX_MAX_WORDS spoken words (the model often adds a short filler like\n    "One moment," before the trigger). Returns one of:\n      - "match": the trigger appears within the prefix window -- the server switches to\n                 capturing the query and never plays this audio.\n      - "maybe": not seen yet, but the window isn\'t exhausted -- keep buffering.\n      - "no":    the window is exhausted with no trigger -- it\'s a normal answer; flush the\n                 buffered audio and speak it.\n    Word-boundary aware (whole spoken words), so \'searching\'/\'researches\' don\'t count.\n    """\n    words = _normalize(text).split()\n    if not words:\n        return "maybe"\n    window = words[:SEARCH_PREFIX_MAX_WORDS]\n    if SEARCH_TRIGGER in window:\n        verdict = "match"\n    elif len(words) > SEARCH_PREFIX_MAX_WORDS:\n        verdict = "no"\n    else:\n        # Special-case a partial last token that could still complete into the trigger\n        # (e.g. "sear"), so we don\'t prematurely give up mid-word on a 1-word utterance.\n        last = words[-1]\n        if len(words) >= SEARCH_PREFIX_MAX_WORDS and not SEARCH_TRIGGER.startswith(last):\n            verdict = "no"\n        else:\n            verdict = "maybe"\n    if SEARCH_DEBUG and verdict in ("match", "no"):\n        print(f"[search_tools] utterance verdict={verdict}: {text.strip()!r}", flush=True)\n    return verdict\n\n\ndef needs_search_response(text: str) -> Optional[str]:\n    """Extract the query from a \'<filler...> <trigger> <query>\' utterance, ignoring any\n    conversational prefix before the trigger. Colon-optional, case-insensitive, and\n    word-boundary aware (the model speaks the trigger, not a written marker)."""\n    match = re.search(\n        rf"\\b{re.escape(SEARCH_TRIGGER)}\\b[\\s:,\\-]*(.+)$",\n        (text or "").strip(),\n        re.IGNORECASE | re.DOTALL,\n    )\n    if not match:\n        return None\n    query = match.group(1).strip().strip(".").strip()\n    return query or None\n\n\n# --- Refusal-detection (the chosen approach) -----------------------------------------\n# Rather than rely on the model emitting a command (unreliable), we watch a COMPLETED\n# agent utterance and, if it looks like a refusal that NAMES the topic, extract that topic\n# as the query. This is inherently fuzzy: some refusals name no topic (we can\'t search\n# then), some name a wrong one. The patterns below are deliberately CONSERVATIVE -- it is\n# better to miss a search than to search garbage and feed the model wrong data. All of this\n# is tunable here in search_tools.py (edit + re-run the write cell + restart the server).\n\n# A refusal / deferral cue -- if present, any "search"/"look up"/"google" in the utterance\n# is being used conversationally ("you could search for X"), not as a command.\n_REFUSAL_CUE_RE = re.compile(\n    r"\\b(i\\s+don\'?t?\\s+have|i\\s+do\\s+not\\s+have|i\\s+can\'?t?|i\\s+cannot|i\\s+couldn\'?t?|"\n    r"not\\s+something\\s+i|you\\s+(?:could|can|might|may|should)|i\'?m\\s+not\\s+sure|"\n    r"no\\s+(?:live|current|real[\\s-]?time)|don\'?t?\\s+have\\s+(?:live|current|real[\\s-]?time|access))\\b",\n    re.IGNORECASE,\n)\n\n# Ordered by reliability. "search for X" is the phrasing our system prompt asks for and is\n# the most trustworthy; "google X" / "look up X" are decent; we intentionally do NOT parse\n# "check <a site>" because it almost never names the actual query. Verb variants\n# (searching/googling/looking) are handled so we catch natural phrasings.\n_REFUSAL_TOPIC_RES = [\n    re.compile(r"\\bsearch(?:ing)?(?:\\s+for)?\\s+(.+)$", re.IGNORECASE),\n    re.compile(r"\\bgoogl(?:e|ing|es)?\\s+(.+)$", re.IGNORECASE),\n    re.compile(r"\\blook(?:ing)?\\s+(?:it\\s+|that\\s+)?up\\s+(.+)$", re.IGNORECASE),\n]\n\n# A captured "topic" starting with one of these is junk ("...on Google or another site"),\n# not a real query.\n_JUNK_TOPIC_START_RE = re.compile(r"^(?:or|and|but|another|other)\\b", re.IGNORECASE)\n\n# Trailing "...online", "...on Google", "...on a finance site", etc. -- strip it off.\n_TRAILING_JUNK_RE = re.compile(\n    r"\\b(online|on\\s+google|on\\s+the\\s+(?:web|internet)|"\n    r"on\\s+(?:a|an|the|some|another)\\s+\\S+(?:\\s+\\S+)?\\s*(?:site|website|app|tracker|page)|"\n    r"elsewhere|yourself|instead|somewhere|right\\s+now|for\\s+(?:the\\s+)?(?:latest|current|exact|real[\\s-]?time)\\b.*)"\n    r"\\b.*$",\n    re.IGNORECASE,\n)\n\n_TOPIC_STOPWORDS = {"it", "that", "this", "them", "those", "there", "here", "one",\n                    "something", "anything", "info", "information", "details"}\n\n\ndef _extract_refusal_topic(text: str) -> Optional[str]:\n    for rx in _REFUSAL_TOPIC_RES:\n        m = rx.search(text)\n        if not m:\n            continue\n        topic = m.group(1).strip()\n        topic = _TRAILING_JUNK_RE.sub("", topic).strip()\n        topic = topic.strip(" .,!?;:\'\\"").strip()\n        if _JUNK_TOPIC_START_RE.match(topic):\n            continue\n        # Drop a leading article / filler word.\n        topic = re.sub(r"^(?:the|a|an|for|about|up|it)\\s+", "", topic, flags=re.IGNORECASE).strip()\n        if not topic or topic.lower() in _TOPIC_STOPWORDS or len(topic) < 3:\n            continue\n        return topic\n    return None\n\n\ndef detect_search_query(utterance_text: str) -> Optional[str]:\n    """Given a COMPLETED agent utterance, return a web-search query, or None.\n\n    Two ways an utterance yields a query:\n      (b) Refusal path (primary): the utterance has a refusal cue AND names a topic to look\n          up ("...you could search for today\'s gold price online" -> "today\'s gold price").\n      (a) Command path (bonus): if the model actually complied with a bare command, i.e. the\n          utterance literally BEGINS with the trigger word ("search today gold price").\n    Returns None for ordinary answers and for refusals that name no topic. The command path\n    is intentionally strict (trigger must be word #1) so ordinary sentences that merely\n    contain "search" ("Let me search my memory ...") are not mistaken for commands.\n    """\n    text = (utterance_text or "").strip()\n    if not text:\n        return None\n    words = _normalize(text).split()\n    if words and words[0] == SEARCH_TRIGGER:\n        query = needs_search_response(text)\n    elif _REFUSAL_CUE_RE.search(text):\n        query = _extract_refusal_topic(text)\n    else:\n        query = None\n    if SEARCH_DEBUG and query:\n        print(f"[search_tools] extracted search query {query!r} from utterance {text!r}", flush=True)\n    return query\n\n\ndef _get_ddgs_client():\n    # The PyPI package was renamed from duckduckgo-search to ddgs in 2025; support both\n    # so this keeps working regardless of which one ends up installed.\n    try:\n        from ddgs import DDGS\n    except ImportError:\n        from duckduckgo_search import DDGS  # type: ignore\n    return DDGS\n\n\ndef _search_duckduckgo(query: str, max_results: int) -> List[Dict[str, str]]:\n    DDGS = _get_ddgs_client()\n    with DDGS() as ddgs:\n        raw = list(ddgs.text(query, max_results=max_results))\n    return [\n        {"title": r.get("title", ""), "snippet": r.get("body", ""), "url": r.get("href", "")}\n        for r in raw\n    ]\n\n\ndef _search_tavily(query: str, max_results: int) -> List[Dict[str, str]]:\n    from tavily import TavilyClient\n\n    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])\n    response = client.search(query, max_results=max_results)\n    return [\n        {"title": r.get("title", ""), "snippet": r.get("content", ""), "url": r.get("url", "")}\n        for r in response.get("results", [])\n    ]\n\n\n_PROVIDERS = {\n    "duckduckgo": _search_duckduckgo,\n    "tavily": _search_tavily,\n}\n\n\ndef search_web(\n    query: str,\n    max_results: Optional[int] = None,\n    provider: Optional[str] = None,\n) -> List[Dict[str, str]]:\n    """Provider-agnostic web search. Returns an explanatory pseudo-result on failure\n    rather than raising, since callers run this mid-conversation and must always be able\n    to build *some* follow-up prompt for Helium."""\n    provider = provider or SEARCH_PROVIDER\n    max_results = max_results or SEARCH_MAX_RESULTS\n    adapter = _PROVIDERS.get(provider)\n    if adapter is None:\n        raise ValueError(f"Unknown search provider \'{provider}\'. Available: {list(_PROVIDERS)}")\n    try:\n        return adapter(query, max_results)\n    except Exception as exc:\n        return [{"title": "", "snippet": f"(search failed: {exc})", "url": ""}]\n\n\ndef _clean_snippet(text: str) -> str:\n    """Strip URLs and website-chrome noise so the injected facts don\'t get read aloud\n    verbatim (the model was echoing snippet fragments like \'you can check it out here\')."""\n    text = re.sub(r"https?://\\S+", "", text)\n    text = re.sub(r"\\bwww\\.\\S+", "", text)\n    # Drop common call-to-action / navigation phrases that snippets are full of (including\n    # an optional leading "you can/could/will" so "You can check it out here" goes entirely).\n    text = re.sub(\n        r"(?i)\\b(?:you\\s+(?:can|could|\'ll|will)\\s+)?"\n        r"(?:check it out here|click here|read more|learn more|see more|"\n        r"visit (?:our|the) (?:site|website|page)|find out more|view (?:all|more))\\b[^.]*\\.?",\n        "",\n        text,\n    )\n    text = re.sub(r"\\s+", " ", text).strip(" .,-|").strip()\n    return text\n\n\ndef _finalize_injection(text: str) -> str:\n    text = text[:SEARCH_PROMPT_MAX_CHARS].rstrip()\n    if INJECT_WITH_SYSTEM_TAGS:\n        return f"<system> {text} <system>"\n    return text\n\n\ndef build_search_prompt(query: str, results: List[Dict[str, str]]) -> str:\n    """Build the text force-fed back into Helium after a search.\n\n    v8: injected as a natural CONTINUATION of the model\'s own speech (no <system> tags by\n    default -- see the INJECT_* config above for why). Structure:\n        <INJECT_PREFIX>  Here is the current information ...: <cleaned facts>.  <INJECT_LEADIN>\n    Everything here is loaded as silent context; the model then SPEAKS its continuation of\n    INJECT_LEADIN out loud, which is the actual answer. Snippets are cleaned/deduped so the\n    model states the fact instead of echoing website chrome (\'you can check it out here\')."""\n    facts = []\n    seen = set()\n    for r in results:\n        snippet = _clean_snippet(r.get("snippet", ""))\n        if not snippet:\n            continue\n        key = snippet[:60].lower()\n        if key in seen:\n            continue\n        seen.add(key)\n        facts.append(snippet)\n\n    if not facts:\n        text = (\n            f"{INJECT_PREFIX} I could not find the current {query}. "\n            f"Tell the user you could not find it right now."\n        )\n        return _finalize_injection(text)\n\n    joined = " ".join(facts)\n    prefix = f"{INJECT_PREFIX} Here is the current information about {query}: "\n    leadin = f". {INJECT_LEADIN}"\n    budget = max(0, SEARCH_PROMPT_MAX_CHARS - len(prefix) - len(leadin) - 20)\n    joined = joined[:budget].rstrip(" .,-|")\n    return _finalize_injection(f"{prefix}{joined}{leadin}")\n'

search_tools_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "search_tools.py"
search_tools_path.write_text(SEARCH_TOOLS_SOURCE, encoding="utf-8")
print(f"Wrote {search_tools_path} ({len(SEARCH_TOOLS_SOURCE)} chars)")


In [8]:
%pip install -q ddgs tavily-python

Note: you may need to restart the kernel to use updated packages.


In [9]:
import importlib.util
import pathlib

# Fail fast on network/dependency issues before spending time loading the (multi-GB)
# model in Section 8 -- same "verify early" spirit as Sections 12/13's smoke tests.
_search_tools_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "search_tools.py"
_spec = importlib.util.spec_from_file_location("search_tools_verify", _search_tools_path)
_search_tools_verify = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_search_tools_verify)

_results = _search_tools_verify.search_web("what is today's date")
assert _results, "search_web() returned no results -- check network access and SEARCH_PROVIDER config."
print(f"Search provider '{_search_tools_verify.SEARCH_PROVIDER}' OK -- got {len(_results)} result(s):")
for r in _results[:2]:
    print(" -", r.get("title") or r.get("url"), "|", r.get("snippet", "")[:80])


Search provider 'duckduckgo' OK -- got 4 result(s):
 - Today's Date and Time - Current Date, Time & More | Find out today's date, current time, day of the week, week number, and more. You
 - What Is Today's Date? — Current Date, Time, Week Number ... | Feb 7, 2026 · See today's exact date and time updated live in every format — ISO


In [10]:
import pathlib

# Adds LMGen.inject_agent_text[_async] -- purely additive, no existing method is
# modified. This is the mechanism that lets server.py force search results into
# Helium's context mid-conversation without resetting streaming state (see the notebook
# markdown above and lm.py's own _step_text_prompt, which this reuses).
lm_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "models" / "lm.py"
_src = lm_path.read_text(encoding="utf-8")

_LM_OLD = '    async def step_system_prompts_async(self, mimi, is_alive: Optional[Callable]=None):\n        await self._step_voice_prompt_async(mimi, is_alive)\n        await self._step_audio_silence_async(is_alive)\n        await self._step_text_prompt_async(is_alive)\n        await self._step_audio_silence_async(is_alive)'

_LM_NEW = '    def inject_agent_text(self, tokens: list[int]):\n        """Force-feed new text into the agent\'s own text channel mid-conversation,\n        without resetting streaming state. Reuses the exact mechanism that loads the\n        initial system prompt (_step_text_prompt) so the LM\'s KV-cache / conversation\n        memory is preserved -- this is what makes mid-call tool-result injection possible\n        for the web-search feature (see search_tools.py and server.py\'s search_state)."""\n        self.text_prompt_tokens = tokens\n        self._step_text_prompt()\n\n    async def inject_agent_text_async(self, tokens: list[int], is_alive: Optional[Callable] = None):\n        self.text_prompt_tokens = tokens\n        await self._step_text_prompt_async(is_alive)\n\n    async def step_system_prompts_async(self, mimi, is_alive: Optional[Callable]=None):\n        await self._step_voice_prompt_async(mimi, is_alive)\n        await self._step_audio_silence_async(is_alive)\n        await self._step_text_prompt_async(is_alive)\n        await self._step_audio_silence_async(is_alive)'

if "def inject_agent_text(" in _src:
    print(f"{lm_path} already patched -- skipping.")
else:
    _count = _src.count(_LM_OLD)
    assert _count == 1, (
        f"Expected exactly 1 match for the step_system_prompts_async anchor in lm.py, found {_count}. "
        "Upstream file may have changed -- update this patch cell."
    )
    lm_path.write_text(_src.replace(_LM_OLD, _LM_NEW), encoding="utf-8")
    print(f"Patched {lm_path} -- added LMGen.inject_agent_text / inject_agent_text_async.")


Patched /workspace/personaplex/moshi/moshi/models/lm.py -- added LMGen.inject_agent_text / inject_agent_text_async.


In [11]:
import pathlib

# Adds the web-search tool-calling interception to server.py's live frame loop. See the
# notebook markdown above for the full design; the short version: Helium's own "SEARCH:
# <query>" reply is caught before its audio reaches the client, search_web() runs, and
# the results are force-fed back into the same live session via LMGen.inject_agent_text_async
# (added to lm.py by the previous cell) so Helium generates the real spoken answer itself.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")


def _apply(text, old, new, label):
    _count = text.count(old)
    assert _count == 1, (
        f"[{label}] expected exactly 1 match while patching server.py, found {_count}. "
        "Upstream file may have changed -- update this patch cell."
    )
    return text.replace(old, new)


if "search_state" in _src:
    print(f"{server_path} already patched -- skipping.")
else:
    _OLD_IMPORTS = 'from .client_utils import make_log, colorize\nfrom .models import loaders, MimiModel, LMModel, LMGen\nfrom .utils.connection import create_ssl_context, get_lan_ip\nfrom .utils.logging import setup_logger, ColorizedLog'
    _NEW_IMPORTS = 'from .client_utils import make_log, colorize\nfrom .models import loaders, MimiModel, LMModel, LMGen\nfrom .search_tools import build_tool_system_prompt, build_search_prompt, needs_search_response, search_web\nfrom .utils.connection import create_ssl_context, get_lan_ip\nfrom .utils.logging import setup_logger, ColorizedLog\n\n\n# --- Web-search tool-calling config (see notebook Section 4b) -------------------------\n# Helium (the PersonaPlex LM) remains the only model in this pipeline; these constants\n# only control the *server-side interception* of its own "SEARCH: <query>" replies, not\n# how the model reasons. See search_tools.py for the search backend itself.\nENABLE_WEB_SEARCH = os.environ.get("PERSONAPLEX_ENABLE_WEB_SEARCH", "1") == "1"\n# There is no ground-truth "utterance boundary" signal in this full-duplex protocol --\n# PAD/EPAD text tokens occur both between agent utterances AND between word-pieces of\n# ongoing speech (text and audio are emitted on independent, delayed schedules). We treat\n# a run of at least this many consecutive PAD frames as "the agent was quiet", and only\n# arm search-detection on the speech that follows such a run. A too-short value costs a\n# little extra buffering latency on false arms (harmless -- buffered audio is still\n# flushed intact); a too-long value can miss a real "SEARCH:" utterance. Tune by observing\n# real conversations if needed.\nUTTERANCE_BOUNDARY_SILENCE_FRAMES = int(os.environ.get("PERSONAPLEX_UTTERANCE_BOUNDARY_SILENCE_FRAMES", "10"))\nSEARCH_DECISION_MAX_FRAMES = 20    # cap on frames spent deciding if an utterance is "SEARCH:"\nSEARCH_CAPTURE_SILENCE_FRAMES = 6  # ~0.5s trailing silence => the query is complete\nSEARCH_CAPTURE_MAX_CHARS = 200     # safety cap on a runaway query capture\nMAX_CONSECUTIVE_SEARCHES = 2       # loop guard: stop auto-searching after N in a row'
    _src = _apply(_src, _OLD_IMPORTS, _NEW_IMPORTS, "imports")

    _OLD_TEXT_PROMPT = '        self.lm_gen.text_prompt_tokens = self.text_tokenizer.encode(wrap_with_system_tags(request.query["text_prompt"])) if len(request.query["text_prompt"]) > 0 else None'
    _NEW_TEXT_PROMPT = '        base_text_prompt = request.query["text_prompt"]\n        combined_text_prompt = build_tool_system_prompt(base_text_prompt) if ENABLE_WEB_SEARCH else base_text_prompt\n        self.lm_gen.text_prompt_tokens = self.text_tokenizer.encode(wrap_with_system_tags(combined_text_prompt)) if len(combined_text_prompt) > 0 else None'
    _src = _apply(_src, _OLD_TEXT_PROMPT, _NEW_TEXT_PROMPT, "text_prompt")

    _OLD_CLOSE_INIT = '        close = False\n        async with self.lock:'
    _NEW_CLOSE_INIT = '        close = False\n        # Per-connection state for the web-search tool-calling interception below.\n        search_state = {\n            "mode": "live",  # live | buffering | capturing\n            "text": "",\n            "frames": [],\n            "silence_run": 0,\n            "consecutive_searches": 0,\n        }\n        async with self.lock:'
    _src = _apply(_src, _OLD_CLOSE_INIT, _NEW_CLOSE_INIT, "close_init")

    _OLD_OPUS_BLOCK = '        async def opus_loop():\n            all_pcm_data = None\n\n            while True:\n                if close:\n                    return\n                await asyncio.sleep(0.001)\n                pcm = opus_reader.read_pcm()\n                if pcm.shape[-1] == 0:\n                    continue\n                if all_pcm_data is None:\n                    all_pcm_data = pcm\n                else:\n                    all_pcm_data = np.concatenate((all_pcm_data, pcm))\n                while all_pcm_data.shape[-1] >= self.frame_size:\n                    be = time.time()\n                    chunk = all_pcm_data[: self.frame_size]\n                    all_pcm_data = all_pcm_data[self.frame_size:]\n                    chunk = torch.from_numpy(chunk)\n                    chunk = chunk.to(device=self.device)[None, None]\n                    codes = self.mimi.encode(chunk)\n                    _ = self.other_mimi.encode(chunk)\n                    for c in range(codes.shape[-1]):\n                        tokens = self.lm_gen.step(codes[:, :, c: c + 1])\n                        if tokens is None:\n                            continue\n                        assert tokens.shape[1] == self.lm_gen.lm_model.dep_q + 1\n                        main_pcm = self.mimi.decode(tokens[:, 1:9])\n                        _ = self.other_mimi.decode(tokens[:, 1:9])\n                        main_pcm = main_pcm.cpu()\n                        opus_writer.append_pcm(main_pcm[0, 0].numpy())\n                        text_token = tokens[0, 0, 0].item()\n                        if text_token not in (0, 3):\n                            _text = self.text_tokenizer.id_to_piece(text_token)  # type: ignore\n                            _text = _text.replace("▁", " ")\n                            msg = b"\\x02" + bytes(_text, encoding="utf8")\n                            await ws.send_bytes(msg)\n                        else:\n                            text_token_map = [\'EPAD\', \'BOS\', \'EOS\', \'PAD\']\n\n        async def send_loop():'
    _NEW_OPUS_BLOCK = '        async def opus_loop():\n            all_pcm_data = None\n\n            while True:\n                if close:\n                    return\n                await asyncio.sleep(0.001)\n                pcm = opus_reader.read_pcm()\n                if pcm.shape[-1] == 0:\n                    continue\n                if all_pcm_data is None:\n                    all_pcm_data = pcm\n                else:\n                    all_pcm_data = np.concatenate((all_pcm_data, pcm))\n                while all_pcm_data.shape[-1] >= self.frame_size:\n                    be = time.time()\n                    chunk = all_pcm_data[: self.frame_size]\n                    all_pcm_data = all_pcm_data[self.frame_size:]\n                    chunk = torch.from_numpy(chunk)\n                    chunk = chunk.to(device=self.device)[None, None]\n                    codes = self.mimi.encode(chunk)\n                    _ = self.other_mimi.encode(chunk)\n                    for c in range(codes.shape[-1]):\n                        tokens = self.lm_gen.step(codes[:, :, c: c + 1])\n                        if tokens is None:\n                            continue\n                        assert tokens.shape[1] == self.lm_gen.lm_model.dep_q + 1\n                        text_token = tokens[0, 0, 0].item()\n                        await handle_agent_frame(tokens, text_token)\n\n        # --- Web-search tool-calling (see notebook Section 4b) --------------------------\n        # Intercepts a "SEARCH: <query>" reply from Helium before its audio ever reaches\n        # the client, runs search_web(), and force-feeds the results back into Helium\'s\n        # own text channel (LMGen.inject_agent_text_async) so it can generate a real,\n        # freshly-sampled spoken answer -- Helium is still the only model writing any\n        # word the user hears; this code only decides *when* to hand it search results.\n        def _decode_frame(tokens):\n            pcm = self.mimi.decode(tokens[:, 1:9])\n            _ = self.other_mimi.decode(tokens[:, 1:9])\n            return pcm.cpu()[0, 0].numpy()\n\n        def _piece_for(text_token):\n            if text_token in (0, 3):\n                return None\n            piece = self.text_tokenizer.id_to_piece(text_token)  # type: ignore\n            return piece.replace("▁", " ")\n\n        async def _send_frame(pcm, piece):\n            opus_writer.append_pcm(pcm)\n            if piece is not None:\n                await ws.send_bytes(b"\\x02" + bytes(piece, encoding="utf8"))\n\n        async def flush_search_buffer():\n            for pcm, piece in search_state["frames"]:\n                await _send_frame(pcm, piece)\n            search_state["frames"] = []\n\n        async def run_search_and_inject(query: str):\n            clog.log("info", f"tool call detected: SEARCH: {query}")\n            loop = asyncio.get_event_loop()\n            results = await loop.run_in_executor(None, search_web, query)\n            prompt_text = build_search_prompt(query, results)\n            clog.log("info", f"injecting search-grounded prompt ({len(prompt_text)} chars)")\n            new_tokens = self.text_tokenizer.encode(wrap_with_system_tags(prompt_text))\n            await self.lm_gen.inject_agent_text_async(new_tokens, is_alive=is_alive)\n            # Mirrors the reset already done after the initial system prompt load below:\n            # frames were deliberately never decoded while capturing the query, so Mimi\'s\n            # own streaming decode state (not the LM\'s) needs a clean point to resume from.\n            self.mimi.reset_streaming()\n            self.other_mimi.reset_streaming()\n\n        async def handle_agent_frame(tokens, text_token):\n            if not ENABLE_WEB_SEARCH:\n                await _send_frame(_decode_frame(tokens), _piece_for(text_token))\n                return\n\n            is_silent = text_token in (0, 3)\n            mode = search_state["mode"]\n\n            if mode == "live":\n                if is_silent:\n                    search_state["silence_run"] += 1\n                    await _send_frame(_decode_frame(tokens), None)\n                    return\n                # A real text piece follows this frame. Only treat it as the start of a\n                # fresh agent utterance (and arm search-detection) if it follows a\n                # sufficiently long pause -- a single PAD frame between word-pieces of\n                # ongoing speech must not re-arm detection mid-sentence.\n                if (search_state["silence_run"] >= UTTERANCE_BOUNDARY_SILENCE_FRAMES\n                        and search_state["consecutive_searches"] < MAX_CONSECUTIVE_SEARCHES):\n                    search_state["mode"] = "buffering"\n                    search_state["text"] = ""\n                    search_state["frames"] = []\n                    search_state["silence_run"] = 0\n                    mode = "buffering"\n                else:\n                    search_state["silence_run"] = 0\n                    await _send_frame(_decode_frame(tokens), _piece_for(text_token))\n                    return\n\n            if mode == "buffering":\n                piece = _piece_for(text_token)\n                search_state["frames"].append((_decode_frame(tokens), piece))\n                if piece:\n                    search_state["text"] += piece\n                stripped = search_state["text"].strip().upper()\n                target = "SEARCH:"\n                if stripped.startswith(target):\n                    search_state["mode"] = "capturing"\n                    search_state["frames"] = []  # this audio is never played\n                elif target.startswith(stripped) and len(search_state["frames"]) < SEARCH_DECISION_MAX_FRAMES:\n                    pass  # still ambiguous (e.g. "SE"); keep buffering, bounded by the cap above\n                else:\n                    await flush_search_buffer()\n                    search_state["mode"] = "live"\n                    search_state["consecutive_searches"] = 0\n                return\n\n            if mode == "capturing":\n                # Audio for this utterance is never decoded or sent -- satisfies the\n                # "hard pause, no filler" behavior while the query is captured/searched.\n                if is_silent:\n                    search_state["silence_run"] += 1\n                else:\n                    search_state["silence_run"] = 0\n                    piece = _piece_for(text_token)\n                    if piece:\n                        search_state["text"] += piece\n                done = (\n                    search_state["silence_run"] >= SEARCH_CAPTURE_SILENCE_FRAMES\n                    or len(search_state["text"]) >= SEARCH_CAPTURE_MAX_CHARS\n                )\n                if done:\n                    query = needs_search_response(search_state["text"])\n                    search_state["mode"] = "live"\n                    search_state["text"] = ""\n                    search_state["silence_run"] = 0\n                    search_state["consecutive_searches"] += 1\n                    if query:\n                        await run_search_and_inject(query)\n                return\n\n        async def send_loop():'
    _src = _apply(_src, _OLD_OPUS_BLOCK, _NEW_OPUS_BLOCK, "opus_block")

    server_path.write_text(_src, encoding="utf-8")
    print(f"Patched {server_path} -- added the web-search tool-calling state machine.")


Patched /workspace/personaplex/moshi/moshi/server.py -- added the web-search tool-calling state machine.


In [ ]:
import pathlib

# One-time upgrade of the already-patched server.py to v2 trigger detection: routes the
# buffering-mode marker match through search_tools.match_search_prefix() (word-boundary
# aware, colon-optional) instead of a hard-coded "SEARCH:" string. After this, ALL future
# prompt/trigger tuning lives in the search_tools.py cell above -- no server.py edits.
# Idempotent: safe to re-run.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")


def _apply(text, old, new, label):
    _c = text.count(old)
    assert _c == 1, (
        f"[{label}] expected exactly 1 match while upgrading server.py, found {_c}. "
        "Make sure the Section 4b server-patch cell ran first (server.py must already "
        "contain the v1 search state machine)."
    )
    return text.replace(old, new)


if "match_search_prefix" in _src:
    print(f"{server_path} already upgraded (v2) -- skipping.")
else:
    _OLD_IMPORT = "from .search_tools import build_tool_system_prompt, build_search_prompt, needs_search_response, search_web"
    _NEW_IMPORT = "from .search_tools import build_tool_system_prompt, build_search_prompt, needs_search_response, search_web, match_search_prefix"
    _src = _apply(_src, _OLD_IMPORT, _NEW_IMPORT, "import")

    _OLD_BUFFERING = '''            if mode == "buffering":
                piece = _piece_for(text_token)
                search_state["frames"].append((_decode_frame(tokens), piece))
                if piece:
                    search_state["text"] += piece
                stripped = search_state["text"].strip().upper()
                target = "SEARCH:"
                if stripped.startswith(target):
                    search_state["mode"] = "capturing"
                    search_state["frames"] = []  # this audio is never played
                elif target.startswith(stripped) and len(search_state["frames"]) < SEARCH_DECISION_MAX_FRAMES:
                    pass  # still ambiguous (e.g. "SE"); keep buffering, bounded by the cap above
                else:
                    await flush_search_buffer()
                    search_state["mode"] = "live"
                    search_state["consecutive_searches"] = 0
                return'''

    _NEW_BUFFERING = '''            if mode == "buffering":
                piece = _piece_for(text_token)
                search_state["frames"].append((_decode_frame(tokens), piece))
                if piece:
                    search_state["text"] += piece
                # Word-boundary-aware, colon-optional trigger detection lives in
                # search_tools.match_search_prefix() so prompt/trigger tuning is one-file.
                verdict = match_search_prefix(search_state["text"])
                if verdict == "match":
                    search_state["mode"] = "capturing"
                    search_state["frames"] = []  # this audio is never played
                elif verdict == "maybe" and len(search_state["frames"]) < SEARCH_DECISION_MAX_FRAMES:
                    pass  # still ambiguous (e.g. "sear"); keep buffering, bounded by the cap
                else:
                    await flush_search_buffer()
                    search_state["mode"] = "live"
                    search_state["consecutive_searches"] = 0
                return'''

    _src = _apply(_src, _OLD_BUFFERING, _NEW_BUFFERING, "buffering")
    server_path.write_text(_src, encoding="utf-8")
    print(f"Upgraded {server_path} -- v2 trigger detection via match_search_prefix().")


In [ ]:
import pathlib

# v3 upgrade: the model tends to say a short filler ("One moment, ...") before the trigger,
# so the buffering window must be long enough to hold that filler plus the trigger before
# deciding. This makes SEARCH_DECISION_MAX_FRAMES env-tunable (default 50 frames ~= 4s) via
# PERSONAPLEX_SEARCH_DECISION_MAX_FRAMES. Idempotent: safe to re-run.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")

_OLD = 'SEARCH_DECISION_MAX_FRAMES = 20    # cap on frames spent deciding if an utterance is "SEARCH:"'
_NEW = ('SEARCH_DECISION_MAX_FRAMES = int(os.environ.get("PERSONAPLEX_SEARCH_DECISION_MAX_FRAMES", "50"))'
        '  # frames to decide if an utterance is a search command (must exceed the filler+trigger span)')

if "PERSONAPLEX_SEARCH_DECISION_MAX_FRAMES" in _src:
    print(f"{server_path} already v3-upgraded -- skipping.")
else:
    _c = _src.count(_OLD)
    assert _c == 1, (
        f"[v3] expected exactly 1 match for the SEARCH_DECISION_MAX_FRAMES line, found {_c}. "
        "Make sure the Section 4b server-patch cell ran first."
    )
    server_path.write_text(_src.replace(_OLD, _NEW), encoding="utf-8")
    print(f"Upgraded {server_path} -- SEARCH_DECISION_MAX_FRAMES now env-tunable (default 50).")


In [ ]:
import pathlib

# v4 fix: the mid-conversation search-result injection reused the pre-handshake is_alive()
# helper, which calls ws.receive(). But recv_loop is already awaiting ws.receive() during
# the live conversation, and aiohttp forbids two concurrent receive() calls (it raised
# "RuntimeError: Concurrent call to receive() is not allowed" and killed the session right
# after the search ran). This swaps in a receive()-free liveness check for the injection
# only (the initial system-prompt load still uses the original is_alive, which is correct
# because recv_loop does not exist yet at that point). Idempotent: safe to re-run.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")

_OLD = '''            new_tokens = self.text_tokenizer.encode(wrap_with_system_tags(prompt_text))
            await self.lm_gen.inject_agent_text_async(new_tokens, is_alive=is_alive)'''

_NEW = '''            new_tokens = self.text_tokenizer.encode(wrap_with_system_tags(prompt_text))
            # IMPORTANT: do NOT reuse the pre-handshake is_alive() here -- it calls
            # ws.receive(), but recv_loop is already awaiting ws.receive() during the live
            # conversation, and aiohttp forbids two concurrent receive() calls. Use a
            # receive()-free liveness check that still yields (asyncio.sleep(0)) so the
            # send/recv loops stay responsive while the grounding prompt is injected.
            async def is_alive_light():
                await asyncio.sleep(0)
                return not (close or ws.closed)
            await self.lm_gen.inject_agent_text_async(new_tokens, is_alive=is_alive_light)'''

if "is_alive_light" in _src:
    print(f"{server_path} already v4-fixed -- skipping.")
else:
    _c = _src.count(_OLD)
    assert _c == 1, (
        f"[v4] expected exactly 1 match for the injection call, found {_c}. "
        "Make sure the Section 4b server-patch cell ran first."
    )
    server_path.write_text(_src.replace(_OLD, _NEW), encoding="utf-8")
    print(f"Fixed {server_path} -- injection uses a receive()-free liveness check (v4).")


In [ ]:
import pathlib

# v5: make text/audio sampling temperature env-configurable. The model SAMPLES its words,
# so at the default text temperature (0.7) it emits the "search ..." trigger only some of
# the time and refuses other times for the same question. Lowering PERSONAPLEX_TEXT_TEMPERATURE
# (set in the config cell) makes instruction-following far more deterministic. Idempotent.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")

_OLD = '''        self.lm_gen.text_prompt_tokens = self.text_tokenizer.encode(wrap_with_system_tags(combined_text_prompt)) if len(combined_text_prompt) > 0 else None'''

_NEW = '''        self.lm_gen.text_prompt_tokens = self.text_tokenizer.encode(wrap_with_system_tags(combined_text_prompt)) if len(combined_text_prompt) > 0 else None
        # Optional sampling-temperature overrides (env-configurable). Lowering the *text*
        # temperature makes the model follow the "reply only: search ..." instruction more
        # deterministically -- the single biggest lever on trigger reliability. It also
        # makes normal conversation less varied, so it is opt-in.
        _text_temp = os.environ.get("PERSONAPLEX_TEXT_TEMPERATURE", "").strip()
        if _text_temp:
            self.lm_gen.temp_text = float(_text_temp)
        _audio_temp = os.environ.get("PERSONAPLEX_AUDIO_TEMPERATURE", "").strip()
        if _audio_temp:
            self.lm_gen.temp = float(_audio_temp)
        clog.log("info", f"text_temp={self.lm_gen.temp_text} audio_temp={self.lm_gen.temp}")'''

if "PERSONAPLEX_TEXT_TEMPERATURE" in _src:
    print(f"{server_path} already v5-upgraded -- skipping.")
else:
    _c = _src.count(_OLD)
    assert _c == 1, (
        f"[v5] expected exactly 1 match for the text_prompt_tokens line, found {_c}. "
        "Make sure the Section 4b server-patch cell ran first."
    )
    server_path.write_text(_src.replace(_OLD, _NEW), encoding="utf-8")
    print(f"Upgraded {server_path} -- text/audio sampling temperature now env-configurable (v5).")


In [ ]:
import pathlib

# v6 (refusal-detection): replace the "buffer + look for a search trigger" state machine
# with a simpler "play audio live, detect intent on the completed utterance" loop. The
# model's refusal is spoken naturally; when it names a topic, we search and inject the
# answer. Normal answers get ZERO added latency (no buffering). Idempotent.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")

_OLD_HAF = '        async def handle_agent_frame(tokens, text_token):\n            if not ENABLE_WEB_SEARCH:\n                await _send_frame(_decode_frame(tokens), _piece_for(text_token))\n                return\n\n            is_silent = text_token in (0, 3)\n            mode = search_state["mode"]\n\n            if mode == "live":\n                if is_silent:\n                    search_state["silence_run"] += 1\n                    await _send_frame(_decode_frame(tokens), None)\n                    return\n                # A real text piece follows this frame. Only treat it as the start of a\n                # fresh agent utterance (and arm search-detection) if it follows a\n                # sufficiently long pause -- a single PAD frame between word-pieces of\n                # ongoing speech must not re-arm detection mid-sentence.\n                if (search_state["silence_run"] >= UTTERANCE_BOUNDARY_SILENCE_FRAMES\n                        and search_state["consecutive_searches"] < MAX_CONSECUTIVE_SEARCHES):\n                    search_state["mode"] = "buffering"\n                    search_state["text"] = ""\n                    search_state["frames"] = []\n                    search_state["silence_run"] = 0\n                    mode = "buffering"\n                else:\n                    search_state["silence_run"] = 0\n                    await _send_frame(_decode_frame(tokens), _piece_for(text_token))\n                    return\n\n            if mode == "buffering":\n                piece = _piece_for(text_token)\n                search_state["frames"].append((_decode_frame(tokens), piece))\n                if piece:\n                    search_state["text"] += piece\n                # Word-boundary-aware, colon-optional trigger detection lives in\n                # search_tools.match_search_prefix() so prompt/trigger tuning is one-file.\n                verdict = match_search_prefix(search_state["text"])\n                if verdict == "match":\n                    search_state["mode"] = "capturing"\n                    search_state["frames"] = []  # this audio is never played\n                elif verdict == "maybe" and len(search_state["frames"]) < SEARCH_DECISION_MAX_FRAMES:\n                    pass  # still ambiguous (e.g. "sear"); keep buffering, bounded by the cap\n                else:\n                    await flush_search_buffer()\n                    search_state["mode"] = "live"\n                    search_state["consecutive_searches"] = 0\n                return\n\n            if mode == "capturing":\n                # Audio for this utterance is never decoded or sent -- satisfies the\n                # "hard pause, no filler" behavior while the query is captured/searched.\n                if is_silent:\n                    search_state["silence_run"] += 1\n                else:\n                    search_state["silence_run"] = 0\n                    piece = _piece_for(text_token)\n                    if piece:\n                        search_state["text"] += piece\n                done = (\n                    search_state["silence_run"] >= SEARCH_CAPTURE_SILENCE_FRAMES\n                    or len(search_state["text"]) >= SEARCH_CAPTURE_MAX_CHARS\n                )\n                if done:\n                    query = needs_search_response(search_state["text"])\n                    search_state["mode"] = "live"\n                    search_state["text"] = ""\n                    search_state["silence_run"] = 0\n                    search_state["consecutive_searches"] += 1\n                    if query:\n                        await run_search_and_inject(query)\n                return'

_NEW_HAF = '        async def handle_agent_frame(tokens, text_token):\n            # v6 (refusal-detection): always play audio live -- no buffering/hold, so normal\n            # answers have zero added latency and the model\'s natural refusal is spoken as\n            # usual. We accumulate the CURRENT utterance\'s text and, when the utterance ends\n            # (a run of silent frames), ask search_tools.detect_search_query() whether it was\n            # a refusal that named a topic (or a bare "search ..." command). If so, we run the\n            # web search and inject the results so the model then speaks the real answer.\n            piece = _piece_for(text_token)\n            await _send_frame(_decode_frame(tokens), piece)\n            if not ENABLE_WEB_SEARCH:\n                return\n\n            if piece is None:  # silent frame\n                if not search_state["text"].strip():\n                    return\n                search_state["silence_run"] += 1\n                if search_state["silence_run"] < UTTERANCE_BOUNDARY_SILENCE_FRAMES:\n                    return\n                # Utterance complete -> evaluate it.\n                utterance = search_state["text"].strip()\n                search_state["text"] = ""\n                search_state["silence_run"] = 0\n                query = detect_search_query(utterance)\n                if query and search_state["consecutive_searches"] < MAX_CONSECUTIVE_SEARCHES:\n                    search_state["consecutive_searches"] += 1\n                    await run_search_and_inject(query)\n                elif query is None:\n                    # An ordinary (non-search) utterance completed -> the model answered\n                    # normally, so clear the loop guard for the next question.\n                    search_state["consecutive_searches"] = 0\n                return\n\n            # Non-silent frame: accumulate the utterance text (bounded).\n            search_state["silence_run"] = 0\n            search_state["text"] += piece\n            if len(search_state["text"]) > 4 * SEARCH_CAPTURE_MAX_CHARS:\n                search_state["text"] = search_state["text"][-4 * SEARCH_CAPTURE_MAX_CHARS:]'

_IMP_OLD = "from .search_tools import build_tool_system_prompt, build_search_prompt, needs_search_response, search_web, match_search_prefix"
_IMP_NEW = "from .search_tools import build_tool_system_prompt, build_search_prompt, needs_search_response, search_web, match_search_prefix, detect_search_query"

if "v6 (refusal-detection)" in _src:
    print(f"{server_path} already v6-upgraded -- skipping.")
else:
    assert _src.count(_IMP_OLD) == 1, (
        "[v6] search_tools import anchor not found -- make sure the v2 server-upgrade cell ran first."
    )
    _src = _src.replace(_IMP_OLD, _IMP_NEW)
    assert _src.count(_OLD_HAF) == 1, (
        "[v6] handle_agent_frame anchor not found/unique -- make sure the Section 4b server "
        "patch and the v2 upgrade cells ran first."
    )
    _src = _src.replace(_OLD_HAF, _NEW_HAF)
    server_path.write_text(_src, encoding="utf-8")
    print(f"Upgraded {server_path} -- handle_agent_frame now uses live-monitor refusal detection (v6).")


In [ ]:
import pathlib

# v8: stop force-wrapping the injected search results in <system> ... <system> tags. A
# <system> block only ever appears at the START of a training conversation, so injecting one
# mid-call reads as "a new conversation is starting" -- the model answered with a greeting/
# sign-off ("Hey, let me know if you have any questions") instead of the actual answer.
# build_search_prompt() now formats the injection itself (as a natural continuation of the
# model's speech), so the server injects its output verbatim. Idempotent.
server_path = pathlib.Path(REPO_DIR) / "moshi" / "moshi" / "server.py"
_src = server_path.read_text(encoding="utf-8")

_OLD = "            new_tokens = self.text_tokenizer.encode(wrap_with_system_tags(prompt_text))"
_NEW = "            new_tokens = self.text_tokenizer.encode(prompt_text)  # build_search_prompt controls the exact inject format (v8)"

if "build_search_prompt controls the exact inject format (v8)" in _src:
    print(f"{server_path} already v8-upgraded -- skipping.")
else:
    _c = _src.count(_OLD)
    assert _c == 1, (
        f"[v8] expected exactly 1 match for the injection encode line, found {_c}. "
        "Make sure the Section 4b server-patch and the v4 fix cells ran first."
    )
    _src = _src.replace(_OLD, _NEW)
    server_path.write_text(_src, encoding="utf-8")
    print(f"Upgraded {server_path} -- injection no longer force-wraps <system> tags (v8).")


## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` (`numpy`, `safetensors`, `huggingface-hub`, `einops`,
`sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`).

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50‑series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake.

In [12]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/." 

Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.8.0+cu128 requires torch==2.8.0, but you have torch 2.4.1 which is incompatible.
torchvision 0.23.0+cu128 requires torch==2.8.0, but you have torch 2.4.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [13]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moshi-personaplex 0.1.0 requires torch<2.5,>=2.2.0, but you have torch 2.12.1+cu130 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [14]:
# `accelerate` enables --cpu-offload (README's "CPU Offload" section); `gradio` enables the optional
# --gradio-tunnel fallback for public exposure if you ever need it instead of RunPod's HTTP proxy.
%pip install -q accelerate gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moshi-personaplex 0.1.0 requires huggingface-hub<0.25,>=0.24, but you have huggingface-hub 1.21.0 which is incompatible.
moshi-personaplex 0.1.0 requires torch<2.5,>=2.2.0, but you have torch 2.12.1+cu130 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.

In [15]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))

Torch version      : 2.12.1+cu130
Torch CUDA version : 13.0
CUDA available     : True
GPU                : NVIDIA GeForce RTX 5090
Compute capability : (12, 0)
bf16 CUDA matmul smoke test OK, result shape: (4096, 4096)


## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.

In [16]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'    
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face Hub.


## 8. Model downloading

`moshi/moshi/server.py` and `moshi/moshi/offline.py` both lazily call `hf_hub_download` for each asset the
first time they need it. We pre-fetch the same files here so that (a) any license/token problem surfaces now
with a clear error instead of mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we
launch the server.

Assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/server.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

In [17]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

config.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

OK  config.json -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/config.json


tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

OK  tokenizer_spm_32k_3.model -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/tokenizer_spm_32k_3.model


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

...

OK  model.safetensors -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/model.safetensors


voices.tgz:   0%|          | 0.00/6.10M [00:00<?, ?B/s]

OK  voices.tgz -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/voices.tgz


dist.tgz:   0%|          | 0.00/598k [00:00<?, ?B/s]

OK  dist.tgz -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/dist.tgz


In [18]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/server.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")

voices.tgz -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/voices (already extracted)
dist.tgz -> /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/dist (already extracted)


## 9. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory. That's appropriate for direct LAN/local access, but on RunPod we're putting the
server behind RunPod's own HTTPS proxy (Section 11), which already terminates TLS at the edge — adding a second,
self-signed TLS layer underneath it would just produce certificate warnings for no benefit. We still create the
directory here so you can flip `USE_APP_TLS = True` above if you'd rather expose `SERVER_PORT` as a raw TCP port
instead of through the HTTP proxy.

In [19]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")

SSL_DIR: /tmp/personaplex_ssl_j9x2p8m8 (only used if USE_APP_TLS = True)


## 9b. (Optional) Build the web client from source — raise the Text Prompt limit

By default `python -m moshi.server` does **not** use the `client/` source in this repo at all: when
`--static` is omitted, `_get_static_path` in `moshi/moshi/server.py` downloads NVIDIA's **prebuilt** web UI
bundle (`dist.tgz`, extracted in Section 8) and serves that instead. That prebuilt bundle caps the "Text
Prompt" box at 1000 characters (`client/src/pages/Queue/Queue.tsx`).

If you plan to give the model a longer topic/grounding prompt (see Section 13b below), run the two cells
below to:
1. Install Node.js, matching `client/.nvmrc`.
2. Patch the cloned repo's `Queue.tsx` to raise that limit to `TEXT_PROMPT_MAX_LEN` characters and add a
   "Topic-only (grounded)" preset.
3. Build the client (`npm ci && npm run build`, per `client/README.md`) and record the build output so
   Section 10 can launch the server with `--static` pointing at it instead of the prebuilt bundle.

Skip this section if a prompt under 1000 characters is enough — the server falls back to NVIDIA's prebuilt
UI automatically when `STATIC_DIR` is left unset.

In [20]:
import shutil
import subprocess

NODE_MAJOR = "20"  # matches client/.nvmrc (v20.12.2)

def _node_major_ok():
    node = shutil.which("node")
    if not node:
        return False
    out = subprocess.run([node, "-v"], capture_output=True, text=True).stdout.strip()
    return out.startswith(f"v{NODE_MAJOR}.")

if _node_major_ok():
    print("Node.js already installed:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())
else:
    !{SUDO}apt-get install -y -qq curl
    !curl -fsSL https://deb.nodesource.com/setup_{NODE_MAJOR}.x | {SUDO}bash -
    !{SUDO}apt-get install -y -qq nodejs
    print("Node.js installed:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())


debconf: delaying package configuration, since apt-utils is not installed
(Reading database ... 49051 files and directories currently installed.)
Preparing to unpack .../libcurl4-openssl-dev_8.5.0-2ubuntu10.10_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (8.5.0-2ubuntu10.10) over (8.5.0-2ubuntu10.6) ...
Preparing to unpack .../curl_8.5.0-2ubuntu10.10_amd64.deb ...
Unpacking curl (8.5.0-2ubuntu10.10) over (8.5.0-2ubuntu10.6) ...
Preparing to unpack .../libcurl4t64_8.5.0-2ubuntu10.10_amd64.deb ...
Unpacking libcurl4t64:amd64 (8.5.0-2ubuntu10.10) over (8.5.0-2ubuntu10.6) ...
Preparing to unpack .../libcurl3t64-gnutls_8.5.0-2ubuntu10.10_amd64.deb ...
Unpacking libcurl3t64-gnutls:amd64 (8.5.0-2ubuntu10.10) over (8.5.0-2ubuntu10.6) ...
Setting up libcurl4t64:amd64 (8.5.0-2ubuntu10.10) ...
Setting up libcurl3t64-gnutls:amd64 (8.5.0-2ubuntu10.10) ...
Setting up libcurl4-openssl-dev:amd64 (8.5.0-2ubuntu10.10) ...
Setting up curl (8.5.0-2ubuntu10.10) ...
Processing triggers for libc-bin (2

In [21]:
import os
import pathlib
import subprocess

TEXT_PROMPT_MAX_LEN = 4000  # raise/lower as needed — see the context-window note in Section 13b below

queue_tsx = pathlib.Path(REPO_DIR) / "client" / "src" / "pages" / "Queue" / "Queue.tsx"
src = queue_tsx.read_text(encoding="utf-8")
patched = src.replace("maxLength={1000}", f"maxLength={{{TEXT_PROMPT_MAX_LEN}}}")
patched = patched.replace("{textPrompt.length}/1000", f"{{textPrompt.length}}/{TEXT_PROMPT_MAX_LEN}")
if patched != src:
    queue_tsx.write_text(patched, encoding="utf-8")
    print(f"Patched {queue_tsx} -> Text Prompt limit is now {TEXT_PROMPT_MAX_LEN} characters.")
else:
    print(f"{queue_tsx}: pattern not found (already patched, or upstream file changed) — left unchanged.")

STATIC_DIR = None
client_dir = os.path.join(REPO_DIR, "client")
try:
    subprocess.run(["npm", "ci"], cwd=client_dir, check=True)
    subprocess.run(["npm", "run", "build"], cwd=client_dir, check=True)
    built_dist = os.path.join(client_dir, "dist")
    assert os.path.isdir(built_dist), f"Expected {built_dist} to exist after `npm run build`."
    STATIC_DIR = built_dist
    print("Custom client built at:", STATIC_DIR)
except Exception as e:
    print(f"Client build failed ({e!r}); falling back to NVIDIA's prebuilt UI (1000-char Text Prompt limit).")
    STATIC_DIR = None


/workspace/personaplex/client/src/pages/Queue/Queue.tsx: pattern not found (already patched, or upstream file changed) — left unchanged.


npm warn deprecated rimraf@3.0.2: Rimraf versions prior to v4 are no longer supported
npm warn deprecated inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.
npm warn deprecated @humanwhocodes/object-schema@2.0.3: Use @eslint/object-schema instead
npm warn deprecated glob@7.2.3: Glob versions prior to v9 are no longer supported
npm warn deprecated @humanwhocodes/config-array@0.13.0: Use @eslint/config-array instead
npm warn deprecated eslint@8.57.1: This version is no longer supported. Please see https://eslint.org/version-support for other options.



added 408 packages, and audited 409 packages in 60s

157 packages are looking for funding
  run `npm fund` for details

19 vulnerabilities (9 moderate, 10 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.


npm notice
npm notice New major version of npm available! 10.8.2 -> 11.18.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.18.0
npm notice To update run: npm install -g npm@11.18.0
npm notice



> kyutai-client@0.0.0 build
> tsc && vite build

vite v5.4.21 building for production...
transforming...


Browserslist: browsers data (caniuse-lite) is 7 months old. Please run:
  npx update-browserslist-db@latest
  Why you should do it regularly: https://github.com/browserslist/update-db#readme



🌼   daisyUI 4.12.24
├─ ✔︎ 2 themes added		https://daisyui.com/docs/themes
╰─ ★ Star daisyUI on GitHub	https://github.com/saadeghi/daisyui

✓ 118 modules transformed.


node_modules/eruda/eruda.js (2:424693): Use of eval in "node_modules/eruda/eruda.js" is strongly discouraged as it poses security risks and may cause issues with minification.


rendering chunks...
computing gzip size...
dist/index.html                                0.64 kB │ gzip:   0.36 kB
dist/assets/audio-processor-BUNQrM5u.js        4.05 kB
dist/assets/decoderWorker.min-BhsGq-5k.js     13.62 kB
dist/assets/encoderWorker.min-DpsJ02BN.js    385.10 kB
dist/assets/index-C8lI51P0.css                29.01 kB │ gzip:   6.43 kB
dist/assets/index-orH2ehow.js              1,276.26 kB │ gzip: 318.52 kB
✓ built in 16.20s
Custom client built at: /workspace/personaplex/client/dist



(!) Some chunks are larger than 500 kB after minification. Consider:
- Using dynamic import() to code-split the application
- Use build.rollupOptions.output.manualChunks to improve chunking: https://rollupjs.org/configuration-options/#output-manualchunks
- Adjust chunk size limit for this warning via build.chunkSizeWarningLimit.


## 10. Backend (+ web UI) startup

This is the repo's **recommended and only documented launch method** — `python -m moshi.server`. It is a
single aiohttp process that serves:
- the WebSocket endpoint `/api/chat` (the real-time speech protocol), and
- the web UI at `/` — either the prebuilt `dist.tgz` bundle (downloaded in Section 8), or your own
  client build from Section 9b if `STATIC_DIR` was set there (passed to the server via `--static`).
  Either way there is **no separate frontend process to start** for normal use.

A notebook cell that calls `web.run_app(...)` directly would block the kernel forever, so we launch it as a
background subprocess and log to a file, then poll that log in the next cell.

In [22]:
import subprocess
import sys

env = os.environ.copy()
LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")

cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT)]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]
if globals().get("STATIC_DIR"):
    cmd += ["--static", STATIC_DIR]

print("Launching:", " ".join(cmd))

log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")

Launching: /usr/local/bin/python -m moshi.server --host 0.0.0.0 --port 8998 --static /workspace/personaplex/client/dist
Server launched with PID 7456. Logs: /workspace/personaplex_server.log


In [23]:
import time

def tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900  # first run downloads + loads a multi-GB model; be generous
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(tail(LOG_PATH))
        raise RuntimeError(
            f"Server process exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(tail(LOG_PATH))

if not ready:
    raise TimeoutError(
        f"Server did not report readiness within {TIMEOUT_S}s. Check the log tail above -- "
        "the most common causes are a slow first-time model download or a CUDA/driver mismatch."
    )

print("\nServer is up and warmed up.")

2026-07-02 13:56:48,867 - __main__ - INFO - retrieving voice prompts
2026-07-02 13:56:49,196 - __main__ - INFO - voice_prompt_dir = /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/voices
2026-07-02 13:56:49,197 - __main__ - INFO - static_path = /workspace/personaplex/client/dist
2026-07-02 13:56:49,411 - __main__ - INFO - loading mimi
2026-07-02 13:57:02,822 - __main__ - INFO - mimi loaded
2026-07-02 13:57:03,268 - __main__ - INFO - loading moshi
2026-07-02 13:57:07,584 - __main__ - INFO - moshi loaded
2026-07-02 13:57:07,600 - __main__ - INFO - warming up the model
2026-07-02 13:57:14,363 - __main__ - INFO - serving static content from /workspace/personaplex/client/dist
2026-07-02 13:57:14,366 - __main__ - INFO - Access the Web UI directly at http://172.21.0.2:8998
======== Running on http://0.0.0.0:8998 ========
(Press CTRL+C to quit)


Server is up and warmed up.


## 11. Expose the port & get the access URL

RunPod will not route external traffic to a port unless you explicitly expose it. In the pod's **Connect**
page, add an **HTTP Service** for `SERVER_PORT` (default `8998`) if you haven't already. RunPod then serves it
at `https://<POD_ID>-<PORT>.proxy.runpod.net`, with RunPod terminating TLS — which is exactly why
`USE_APP_TLS = False` is the right default here (see Section 9).

In [24]:
pod_id = os.environ.get("RUNPOD_POD_ID")

if pod_id:
    public_url = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):")
    print(" ", public_url)
else:
    print(
        "RUNPOD_POD_ID was not found in the environment. If you are on RunPod, expose "
        f"port {SERVER_PORT} as an HTTP Service from the pod's Connect page and use the proxy URL shown there."
    )

scheme = "https" if USE_APP_TLS else "http"
print(f"Local URL inside the pod: {scheme}://localhost:{SERVER_PORT}")

RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):
  https://zybere5xfw27ny-8998.proxy.runpod.net
Local URL inside the pod: http://localhost:8998


## 12. Verification — offline inference smoke test

This runs the repo's documented `moshi.offline` "Assistant example" exactly as given in `README.md` — it
streams `assets/test/input_assistant.wav` through the model with voice prompt `NATF2.pt` and writes an output
WAV + JSON transcript. This doesn't need a browser, microphone, or open port, so it's a good way to confirm the
backend, weights and GPU are all working correctly before testing the live UI.

In [ ]:
import subprocess
import sys

offline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", "NATF2.pt",
    "--input-wav", "assets/test/input_assistant.wav",
    "--seed", "42424242",
    "--output-wav", "output_assistant.wav",
    "--output-text", "output_assistant.json",
]

result = subprocess.run(
    offline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True,
)

print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Offline smoke test failed -- see output above.")

print("\nOffline smoke test succeeded.")

In [ ]:
import json

from IPython.display import Audio, display

output_wav_path = os.path.join(REPO_DIR, "output_assistant.wav")
output_json_path = os.path.join(REPO_DIR, "output_assistant.json")

display(Audio(output_wav_path))

with open(output_json_path) as f:
    tokens = json.load(f)
print("Generated text tokens (joined):")
print("".join(tokens))

## 13. Verification — HTTP check on the live server

Confirms the running server process answers on `/` (the web UI's `index.html`, served from `dist.tgz`).

In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None

with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    print("HTTP status :", resp.status)
    print("Content-Type:", resp.headers.get("Content-Type"))
    assert resp.status == 200, f"Expected 200, got {resp.status}"

print("Web UI is being served correctly.")

## 13b. Provide topic information for grounded responses

PersonaPlex exposes exactly one channel for "give it information, then make it stick only to that": the
**Text Prompt** field already in the web UI. That field *is* the model's system prompt — `server.py` wraps
whatever you type in `<system> ... <system>` tags (`wrap_with_system_tags`) and feeds it to the model token
by token as the very first thing in its context, before either of you say a word
(`moshi/moshi/models/lm.py: step_system_prompts`). There is no separate "context" field or knowledge base —
everything goes through this one box.

**Where to put it:** type/paste it into the Text Prompt box in Section 14 below, *before* you click connect.
Anything you only say out loud after connecting is ordinary spoken conversation, not a system prompt — it
is not privileged the way text typed into that box is.

**How to phrase it:** PersonaPlex's own training data already has a pattern for exactly this — a short
role sentence followed by `Information: <facts>` (see the "Customer Service Roles" examples in
`README.md`). That pattern is in-distribution for the model, so it follows it more reliably than a generic
"only answer from this text" instruction wrapped around a long free-form document. The **"Topic-only
(grounded)"** preset added to the web UI (`client/src/pages/Queue/Queue.tsx`, only present if you built the
client in Section 9b) gives you a starting template — replace its bracketed placeholders with your topic
and facts. The cell below builds and prints the same kind of string for you to copy in.

**Limits this does not remove:**
1. This is a soft instruction a 7B model tries to follow, not a hard filter. There is no
   retrieval/grounding-enforcement layer in this codebase, so it can still drift, guess, or blend in outside
   knowledge — "strict" here is best-effort prompting, not a guarantee.
2. The model's causal attention only spans the last `context = 3000` steps
   (`moshi/moshi/models/loaders.py`) at `FRAME_RATE = 12.5` Hz — about 4 minutes of combined
   prompt-loading + silence + conversation. Every character of your Text Prompt is loaded as its own step
   before the conversation starts and counts against that same 4-minute budget: a longer prompt means a
   longer pause before you can start talking, *and* less of the window left before the model can no longer
   attend back to your original prompt at all (it falls out of the attention window like anything else
   `context` steps in the past). There is no mechanism in this code to re-inject the prompt mid-conversation
   — for sessions longer than a few minutes, treat the grounding as something that fades, not something
   permanent.
3. Given (2), prefer a tight list of concrete facts (closer to the README's examples) over a long prose
   document, even though Section 9b raises the box's limit up to `TEXT_PROMPT_MAX_LEN` characters.

**Interaction with Section 4b (live web search):** if you enabled `ENABLE_WEB_SEARCH` in Section 4b,
the server automatically appends its own tool-use instructions to whatever you type here (or even if
you leave this box empty) -- you do not need to mention search/tools yourself. Those instructions
also consume a small slice of the same character/attention-window budget described below, so if
you're already writing a long, tightly-budgeted grounding prompt, account for a few hundred extra
characters, or set `ENABLE_WEB_SEARCH = False` in Section 4b if you'd rather keep the full budget for
static grounding text only.

In [ ]:
CUSTOM_TOPIC_INFO = (
    "Replace this with your facts, e.g.: Our return policy allows returns within 30 days with a receipt. "
    "Store hours are 9am-6pm Monday-Saturday, closed Sunday."
)
ROLE_SENTENCE = "You are a knowledgeable assistant for [your topic]."

grounded_prompt = (
    f"{ROLE_SENTENCE} Only use the Information below to answer questions; if something is not covered by "
    f"it, say it is outside what you were told and you cannot help with that. Do not use any other "
    f"knowledge. Information: {CUSTOM_TOPIC_INFO}"
)

limit = globals().get("TEXT_PROMPT_MAX_LEN", 1000)
print(f"Characters: {len(grounded_prompt)} (Text Prompt box limit: {limit})")
print()
print(grounded_prompt)
print()
print("Copy the text above into the 'Text Prompt' box in Section 14, before clicking connect.")


## 14. Using the live web UI

Open the URL printed in Section 11 in a browser, allow microphone access when prompted (this is the one
browser permission step that can't be automated), and start talking.

### Voices (from `README.md`)

```
Natural(female): NATF0, NATF1, NATF2, NATF3
Natural(male):   NATM0, NATM1, NATM2, NATM3
Variety(female): VARF0, VARF1, VARF2, VARF3, VARF4
Variety(male):   VARM0, VARM1, VARM2, VARM3, VARM4
```

### Example role prompts (from `README.md`)

- Assistant role: `You are a wise and friendly teacher. Answer questions or provide advice in a clear and
  engaging way.`
- Casual conversation: `You enjoy having a good conversation.`
- Customer service example: `You work for CitySan Services which is a waste management company and your name
  is Ayelen Lucero. Information: Verify customer name Omar Torres. Current schedule: every other week.
  Upcoming pickup: April 12th. Compost bin service available for $8/month add-on.`

See the repo's `README.md` "Prompting Guide" section for more examples and guidance.

### Live web search (tool-calling)

If Section 4b's `ENABLE_WEB_SEARCH` is on, you can also ask things like:

- "What's today's gold price in Bangladesh?"
- "What's the weather like in Dhaka right now?"
- "What's the latest news about a topic you care about?"
- "What's the USD to BDT exchange rate today?"

Helium decides for itself whether a question needs a search -- ordinary questions get an immediate
answer as before. For a live-info question, expect a short **silent pause** (a few seconds, no
filler audio) while it searches and reads the results back to itself, then it answers normally in
its own voice, in the same session. See Section 4b for exactly how this works and its limits.

## 15. GPU optimization notes for RTX 5090

- **bf16 by default**: `moshi/moshi/models/loaders.py:get_moshi_lm` loads the LM in `torch.bfloat16`. Blackwell
  has fast native bf16 Tensor Core throughput, so no dtype changes are needed.
- **`--cpu-offload` is unnecessary here**: it exists in `server.py`/`offline.py` for GPUs with insufficient
  VRAM (via `accelerate`'s `infer_auto_device_map`). An RTX 5090's 32GB comfortably fits this 7B model plus the
  Mimi codec; only enable `USE_CPU_OFFLOAD` if you're also running other large workloads on the same GPU.
- **One process = one GPU's worth of model**: `ServerState.__init__` loads two Mimi instances and the LM once
  per process and keeps them resident (`streaming_forever`). Don't launch a second `moshi.server` process
  against the same GPU unless you've confirmed there's VRAM headroom — check with `nvidia-smi`.
- **Warmup cost is already handled**: `state.warmup()` runs 4 dummy frames through the full encode → LM →
  decode path right after model load, ahead of any real connections — that's the per-process latency you saw
  the log wait for in Section 10, not something to optimize further.
- **CUDA build must match the GPU**: Blackwell (`sm_120`) needs the `cu130` PyTorch wheels installed in
  Section 5. If you ever see `no kernel image is available for execution on the device`, that cell needs to be
  re-run (something likely reinstalled a different torch build afterward).

## 16. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `401`/`403` downloading model assets | License not accepted, or `HF_TOKEN` doesn't belong to the account that accepted it | Re-check Section 7: accept the license at the model page with the **same** account whose token you pasted |
| `no kernel image is available for execution on the device` | Torch build doesn't have Blackwell (`sm_120`) kernels | Re-run the `cu130` reinstall cell in Section 5, then re-run Section 6's smoke test |
| `ImportError` / build errors mentioning `opus` while installing `sphn` | `libopus-dev` missing | Re-run Section 3, then re-run Section 5 |
| Server process exits immediately, log shows a CUDA OOM | Not enough free VRAM (e.g. another process holds the GPU) | Check `nvidia-smi`; consider `USE_CPU_OFFLOAD = True` (Section 2) and re-run Sections 10‑11 |
| Browser blocks microphone / `getUserMedia` fails | Page wasn't loaded over a secure context | Use the RunPod **proxy** URL from Section 11 (HTTPS at the edge), not a plain `http://<pod-ip>:8998` URL |
| Can't reach the URL from outside the pod at all | Port not exposed | In the RunPod console, add `SERVER_PORT` as an HTTP Service on the pod's Connect page |
| First launch seems to hang for several minutes | Normal — first run downloads multi-GB weights and extracts `voices.tgz`/`dist.tgz` | Watch the log tail printed by Section 10; increase `TIMEOUT_S` if your network is slow |
| `mkcert` warnings in the log | Only relevant when `USE_APP_TLS = True`; `moshi/moshi/utils/connection.py` falls back to plain HTTP automatically if `mkcert` can't be installed | Safe to ignore in default (proxy) mode |

### Recap: what genuinely cannot be automated by this notebook
1. Clicking "Agree and access repository" on the gated HF model page.
2. Issuing the HF access token for that account.
3. Exposing `SERVER_PORT` as an HTTP Service in the RunPod console.
4. The browser's microphone-permission prompt.

## 17. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow — running it will terminate the
backend you just verified above. Run it deliberately when you're done with the session, not as part of a
top-to-bottom "Run All".

In [ ]:
# Intentionally NOT meant to run automatically as part of the startup sequence above.
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Server process {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Server process {server_proc.pid} killed after not stopping gracefully.")